# Project Completion Local + Vertex Notebook

This notebook keeps the working path small: data-processing SQL is imported from `project_completion/data_processing.py`, the trainer can run locally from the notebook, and the same package can be submitted as a Vertex AI Custom Job. CI/CD files are unchanged. Environment values are read from shell variables or `.env`.

## Environment Setup

Run this once in a fresh local or Workbench environment.

In [1]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-local.txt"])

# Required before Vertex submit or GCS package upload:
#   gcloud auth application-default login
#   gcloud auth login
#   gcloud config set project $PROJECT_ID

import os
from pathlib import Path

env_candidates = [Path(".env"), Path("env"), Path("env.txt"), Path(".env.txt")]
env_file = next((path for path in env_candidates if path.exists()), None)

if env_file:
    print("Loading environment from:", env_file)
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
else:
    print("No env file found. Expected one of:", [str(path) for path in env_candidates])

required_env = ["PROJECT_ID", "REGION", "BUCKET"]
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    print("Missing env vars:", missing_env)
    print("Upload env.example or export these variables before running Vertex/GCS cells.")
else:
    print("Environment ready:", {name: os.getenv(name) for name in required_env})


Loading environment from: env.txt
Environment ready: {'PROJECT_ID': 'qwiklabs-asl-03-7c1aaee9a503', 'REGION': 'us-central1', 'BUCKET': 'qwiklabs-asl-03-7c1aaee9a503'}


In [2]:
import glob
import math
import os
import subprocess
from datetime import datetime

from project_completion.data_processing import (
    DataProcessingConfig,
    build_export_jobs,
    build_feature_sql,
    build_preview_query,
    build_split_sql,
)

## Configuration

In [3]:
def get_gcloud_project(default="qwiklabs-asl-03-7c1aaee9a503"):
    try:
        value = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"],
            stderr=subprocess.DEVNULL,
            text=True,
        ).strip()
        return value or default
    except Exception:
        return default


PROJECT_ID = os.getenv("PROJECT_ID", get_gcloud_project())
REGION = os.getenv("REGION", "us-central1")
BUCKET = os.getenv("BUCKET", PROJECT_ID)
BUCKET_URI = f"gs://{BUCKET}"

config = DataProcessingConfig(
    project_id=PROJECT_ID,
    dataset_id=os.getenv("DATASET_ID", "jindong_lin"),
    table_id=os.getenv("TABLE_ID", "project_data"),
    bucket_uri=os.getenv("DATA_GCS_BASE_URI", f"{BUCKET_URI}/jindong_lin/data"),
)

TRAIN_DATA_PATH = f"{config.data_gcs_prefix}/train/project-train-*"
EVAL_DATA_PATH = f"{config.data_gcs_prefix}/valid/project-valid-*"
LOCAL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/local_embedding_dnn_test"

print("PROJECT_ID:", PROJECT_ID)
print("REGION:", REGION)
print("TRAIN_DATA_PATH:", TRAIN_DATA_PATH)
print("EVAL_DATA_PATH:", EVAL_DATA_PATH)
print("LOCAL_OUTPUT_DIR:", LOCAL_OUTPUT_DIR)

PROJECT_ID: qwiklabs-asl-03-7c1aaee9a503
REGION: us-central1
TRAIN_DATA_PATH: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*
EVAL_DATA_PATH: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*
LOCAL_OUTPUT_DIR: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_embedding_dnn_test


## Data Processing Smoke Test

In [4]:
preview_query = build_preview_query(config.source_table)
feature_sql = build_feature_sql(config.source_table, config.feature_table)
split_sql = build_split_sql(config.feature_table, config.train_table, config.valid_table, config.test_table)
export_jobs = build_export_jobs(config)

assert f"FROM `{config.source_table}`" in preview_query
assert "DATE_DIFF(DATE(S90), DATE(S30), DAY) AS days_to_S90" in feature_sql
assert "FARM_FINGERPRINT(CONCAT(SID, '-', PID))" in split_sql
assert export_jobs[0][1].endswith("/train/project-train-*.csv")
assert export_jobs[1][1].endswith("/valid/project-valid-*.csv")

print("Data-processing helper smoke test passed.")

Data-processing helper smoke test passed.


## Optional Data Processing

Keep `RUN_DATA_PROCESSING = False` for normal training runs. Set it to `True` only for the first run or when the source BigQuery table has changed and the GCS CSV files must be regenerated.

In [5]:
RUN_DATA_PROCESSING = os.getenv("RUN_DATA_PROCESSING", "False").lower() == "true"

if RUN_DATA_PROCESSING:
    from google.cloud import bigquery, storage
    from project_completion.data_processing import export_tables_to_gcs, run_bigquery_data_processing

    bq_client = bigquery.Client(project=PROJECT_ID)
    storage_client = storage.Client(project=PROJECT_ID)
    bq_location = os.getenv("BQ_LOCATION")
    if not bq_location:
        dataset_ref = bigquery.DatasetReference(PROJECT_ID, config.dataset_id)
        bq_location = bq_client.get_dataset(dataset_ref).location

    row_counts = run_bigquery_data_processing(bq_client, config, location=bq_location)
    export_tables_to_gcs(
        bq_client,
        config,
        location=bq_location,
        storage_client=storage_client,
        clean_existing=True,
    )
    print("Regenerated CSV files from BigQuery.")
    print(row_counts)
else:
    print("Skip data processing. Reusing existing CSV files in GCS:")
    print("  train:", TRAIN_DATA_PATH)
    print("  valid:", EVAL_DATA_PATH)


Skip data processing. Reusing existing CSV files in GCS:
  train: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*
  valid: gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*


## Run the trainer locally from the notebook

In [6]:
def _env_int(name, default):
    return int(os.getenv(name, str(default)))


def _env_float(name, default):
    return float(os.getenv(name, str(default)))


def _resolve_steps(env_name, table_name, batch_size, fallback, max_steps=None):
    raw_value = os.getenv(env_name, "AUTO")
    if raw_value.upper() != "AUTO":
        return int(raw_value)

    try:
        from google.cloud import bigquery
        bq_client_for_counts = bigquery.Client(project=PROJECT_ID)
        rows = bq_client_for_counts.get_table(table_name).num_rows
        if rows == 0:
            raise ValueError(f"{table_name} has 0 rows")
        steps = max(1, math.ceil(rows / batch_size))
        if max_steps is not None:
            steps = min(max_steps, steps)
        print(f"{env_name}=AUTO -> {steps} steps from {rows} rows in {table_name}")
        return steps
    except Exception as exc:
        print(f"{env_name}=AUTO could not read BigQuery row count: {exc}")
        print(f"Falling back to {fallback}. Set {env_name} explicitly in .env to override.")
        return fallback


LOCAL_BATCH_SIZE = _env_int("LOCAL_BATCH_SIZE", 64)
LOCAL_EPOCHS = _env_int("LOCAL_EPOCHS", 10)
LOCAL_TRAIN_STEPS_PER_EPOCH = _resolve_steps(
    "LOCAL_TRAIN_STEPS_PER_EPOCH",
    config.train_table,
    LOCAL_BATCH_SIZE,
    fallback=1,
    max_steps=_env_int("LOCAL_MAX_TRAIN_STEPS_PER_EPOCH", 100),
)
LOCAL_VALIDATION_STEPS = _resolve_steps(
    "LOCAL_VALIDATION_STEPS",
    config.valid_table,
    LOCAL_BATCH_SIZE,
    fallback=1,
    max_steps=_env_int("LOCAL_MAX_VALIDATION_STEPS", 50),
)
HIDDEN_UNITS = os.getenv("HIDDEN_UNITS", "128 64 32")
LEARNING_RATE = _env_float("LEARNING_RATE", 0.001)
DROPOUT_RATE = _env_float("DROPOUT_RATE", 0.2)
LABEL_SCALE = _env_float("LABEL_SCALE", 365.0)

print("Local batch size:", LOCAL_BATCH_SIZE)
print("Local epochs:", LOCAL_EPOCHS)
print("Local train_steps_per_epoch:", LOCAL_TRAIN_STEPS_PER_EPOCH)
print("Local validation_steps:", LOCAL_VALIDATION_STEPS)

local_train_cmd = [
    "python", "-m", "trainer.task",
    "--train_data_path", TRAIN_DATA_PATH,
    "--eval_data_path", EVAL_DATA_PATH,
    "--output_dir", LOCAL_OUTPUT_DIR,
    "--batch_size", str(LOCAL_BATCH_SIZE),
    "--epochs", str(LOCAL_EPOCHS),
    "--train_steps_per_epoch", str(LOCAL_TRAIN_STEPS_PER_EPOCH),
    "--validation_steps", str(LOCAL_VALIDATION_STEPS),
    "--hidden_units", HIDDEN_UNITS,
    "--learning_rate", str(LEARNING_RATE),
    "--dropout_rate", str(DROPOUT_RATE),
    "--label_scale", str(LABEL_SCALE),
]

env = os.environ.copy()
env.setdefault("CUDA_VISIBLE_DEVICES", "-1")
env.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

subprocess.run(local_train_cmd, cwd="project_completion", env=env, check=True)


LOCAL_TRAIN_STEPS_PER_EPOCH=AUTO -> 100 steps from 94913 rows in qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_train
LOCAL_VALIDATION_STEPS=AUTO -> 50 steps from 11850 rows in qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_valid
Local batch size: 64
Local epochs: 10
Local train_steps_per_epoch: 100
Local validation_steps: 50


2026-06-10 16:05:22.697596: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-10 16:05:22.726660: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-10 16:05:22.735158: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Embedding config: {
  "RID": {
    "vocabulary_size": 4,
    "embedding_dim": 2
  },
  "ZDP": {
    "vocabulary_size": 151,
    "embedding_dim": 13
  },
  "GID": {
    "vocabulary_size": 12,
    "embedding_dim": 4
  },
  "MID": {
    "vocabulary_size": 16,
    "embedding_dim": 4
  }
}
Epoch 1/10
100/100 - 7s - 66ms/step - loss: 0.6022 - scaled_mae: 0.4246 - scaled_rmse: 0.7760 - val_loss: 0.3376 - val_scaled_mae: 0.2620 - val_scaled_rmse: 0.5810
Epoch 2/10
100/100 - 1s - 13ms/step - loss: 0.3274 - scaled_mae: 0.3012 - scaled_rmse: 0.5721 - val_loss: 0.4215 - val_scaled_mae: 0.2723 - val_scaled_rmse: 0.6492
Epoch 3/10
100/100 - 2s - 16ms/step - loss: 0.3161 - scaled_mae: 0.2884 - scaled_rmse: 0.5623 - val_loss: 0.1427 - val_scaled_mae: 0.2324 - val_scaled_rmse: 0.3777
Epoch 4/10
100/100 - 3s - 29ms/step - loss: 0.3110 - scaled_mae: 0.2890 - scaled_rmse: 0.5577 - val_loss: 0.0923 - val_scaled_mae: 0.1802 - val_scaled_rmse: 0.3038
Epoch 5/10
100/100 - 1s - 15ms/step - loss: 0.3154 - scale

CompletedProcess(args=['python', '-m', 'trainer.task', '--train_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/train/project-train-*', '--eval_data_path', 'gs://qwiklabs-asl-03-7c1aaee9a503/jindong_lin/data/project_completion/valid/project-valid-*', '--output_dir', 'gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_embedding_dnn_test', '--batch_size', '64', '--epochs', '10', '--train_steps_per_epoch', '100', '--validation_steps', '50', '--hidden_units', '128 64 32', '--learning_rate', '0.001', '--dropout_rate', '0.2', '--label_scale', '365.0'], returncode=0)

## View Local Training with TensorBoard

Run this cell after local training finishes to inspect TensorBoard logs written under `LOCAL_OUTPUT_DIR/tensorboard`.


In [7]:
# TensorBoard for the local trainer output.
# If the extension is already loaded, Jupyter may print a harmless message.
%load_ext tensorboard

TENSORBOARD_LOGDIR = f"{LOCAL_OUTPUT_DIR.rstrip('/')}/tensorboard"
print("TensorBoard logdir:", TENSORBOARD_LOGDIR)
%tensorboard --logdir {TENSORBOARD_LOGDIR} --port 8000


TensorBoard logdir: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/local_embedding_dnn_test/tensorboard


## Submit Custom Job using the Vertex AI SDK

This section creates or reuses a Vertex AI TensorBoard instance by default, gets the compute service account with `gcloud`, and submits the Custom Job with both `service_account` and `tensorboard`.


In [8]:
from google.cloud import aiplatform

RUN_VERTEX_CUSTOM_JOB = True

# Get the compute service account from the current GCP project.
GCP_SRV_ACC = !gcloud iam service-accounts list --filter="email ~ -compute@developer.gserviceaccount.com" --format="value(email)"
if not GCP_SRV_ACC:
    raise ValueError("No compute service account found. Check gcloud project/account configuration.")
GCP_SERV_ACCOUNT = GCP_SRV_ACC[0]
print("GCP_SERV_ACCOUNT:", GCP_SERV_ACCOUNT)

# Create or reuse a Vertex AI TensorBoard instance.
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=f"{BUCKET_URI}/project_completion/staging")

TB_DISPLAY_NAME = "project-forcast-tensorboard"

existing_tensorboards = aiplatform.Tensorboard.list(
    filter=f"display_name={TB_DISPLAY_NAME}",
    project=PROJECT_ID,
    location=REGION,
)

if existing_tensorboards:
    print(f"Found existing TensorBoard instance: {TB_DISPLAY_NAME}")
    tensorboard = existing_tensorboards[0]
else:
    print(f"TensorBoard instance '{TB_DISPLAY_NAME}' not found. Creating a new one...")
    tensorboard = aiplatform.Tensorboard.create(
        display_name=TB_DISPLAY_NAME,
        project=PROJECT_ID,
        location=REGION,
    )

TB_RESOURCE_NAME = tensorboard.resource_name
print(f"TB_RESOURCE_NAME: {TB_RESOURCE_NAME}")

VERTEX_BATCH_SIZE = _env_int("VERTEX_BATCH_SIZE", 64)
VERTEX_EPOCHS = _env_int("VERTEX_EPOCHS", 10)
VERTEX_TRAIN_STEPS_PER_EPOCH = _resolve_steps(
    "VERTEX_TRAIN_STEPS_PER_EPOCH",
    config.train_table,
    VERTEX_BATCH_SIZE,
    fallback=1,
    max_steps=None,
)
VERTEX_VALIDATION_STEPS = _resolve_steps(
    "VERTEX_VALIDATION_STEPS",
    config.valid_table,
    VERTEX_BATCH_SIZE,
    fallback=1,
    max_steps=None,
)

print("Vertex batch size:", VERTEX_BATCH_SIZE)
print("Vertex epochs:", VERTEX_EPOCHS)
print("Vertex train_steps_per_epoch:", VERTEX_TRAIN_STEPS_PER_EPOCH)
print("Vertex validation_steps:", VERTEX_VALIDATION_STEPS)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_OUTPUT_DIR = f"{BUCKET_URI}/project_completion/trained_dnn_model_{timestamp}"
PACKAGE_URI = os.getenv("PACKAGE_URI") or f"{BUCKET_URI}/project_completion/packages/project_completion_trainer_{timestamp}.tar.gz"
TRAINING_IMAGE_URI = os.getenv(
    "TRAINING_IMAGE_URI",
    "us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
)

subprocess.run(["python", "setup.py", "sdist", "--formats=gztar"], cwd="project_completion", check=True)
local_package = sorted(glob.glob("project_completion/dist/*.tar.gz"))[-1]
subprocess.run(["gcloud", "storage", "cp", local_package, PACKAGE_URI], check=True)

worker_pool_specs = [
    {
        "machine_spec": {"machine_type": os.getenv("VERTEX_MACHINE_TYPE", "n1-standard-4")},
        "replica_count": 1,
        "python_package_spec": {
            "executor_image_uri": TRAINING_IMAGE_URI,
            "package_uris": [PACKAGE_URI],
            "python_module": "trainer.task",
            "args": [
                "--train_data_path", TRAIN_DATA_PATH,
                "--eval_data_path", EVAL_DATA_PATH,
                "--output_dir", MODEL_OUTPUT_DIR,
                "--batch_size", str(VERTEX_BATCH_SIZE),
                "--epochs", str(VERTEX_EPOCHS),
                "--train_steps_per_epoch", str(VERTEX_TRAIN_STEPS_PER_EPOCH),
                "--validation_steps", str(VERTEX_VALIDATION_STEPS),
                "--hidden_units", HIDDEN_UNITS,
                "--learning_rate", str(LEARNING_RATE),
                "--dropout_rate", str(DROPOUT_RATE),
                "--label_scale", str(LABEL_SCALE),
            ],
        },
    }
]

job = aiplatform.CustomJob(
    display_name=f"project_completion_dnn_training_{timestamp}",
    worker_pool_specs=worker_pool_specs,
    base_output_dir=MODEL_OUTPUT_DIR,
)

if RUN_VERTEX_CUSTOM_JOB:
    job.submit(
        service_account=GCP_SERV_ACCOUNT,
        tensorboard=TB_RESOURCE_NAME,
    )
    print("Submitted Custom Job:", job.resource_name)
    print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)
    print("Vertex AI TensorBoard:", TB_RESOURCE_NAME)
else:
    print("Custom Job is configured but not submitted. Set RUN_VERTEX_CUSTOM_JOB=True to submit.")
    print("PACKAGE_URI:", PACKAGE_URI)
    print("MODEL_OUTPUT_DIR:", MODEL_OUTPUT_DIR)
    print("Vertex AI TensorBoard:", TB_RESOURCE_NAME)



/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


GCP_SERV_ACCOUNT: 192763454488-compute@developer.gserviceaccount.com
Found existing TensorBoard instance: project-forcast-tensorboard
TB_RESOURCE_NAME: projects/192763454488/locations/us-central1/tensorboards/4361791214689714176
VERTEX_TRAIN_STEPS_PER_EPOCH=AUTO -> 1484 steps from 94913 rows in qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_train
VERTEX_VALIDATION_STEPS=AUTO -> 186 steps from 11850 rows in qwiklabs-asl-03-7c1aaee9a503.jindong_lin.project_completion_valid
Vertex batch size: 64
Vertex epochs: 10
Vertex train_steps_per_epoch: 1484
Vertex validation_steps: 186
running sdist
running egg_info
writing project_completion_trainer.egg-info/PKG-INFO
writing dependency_links to project_completion_trainer.egg-info/dependency_links.txt
writing requirements to project_completion_trainer.egg-info/requires.txt
writing top-level names to project_completion_trainer.egg-info/top_level.txt
reading manifest file 'project_completion_trainer.egg-info/SOURCES.txt'
writing manifest


Copying file://project_completion/dist/project_completion_trainer-0.1.tar.gz to gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/packages/project_completion_trainer_20260610_160652.tar.gz
  
.


Creating CustomJob
CustomJob created. Resource name: projects/192763454488/locations/us-central1/customJobs/257110881370898432
To use this CustomJob in another session:
custom_job = aiplatform.CustomJob.get('projects/192763454488/locations/us-central1/customJobs/257110881370898432')
View Custom Job:
https://console.cloud.google.com/agent-platform/locations/us-central1/training/257110881370898432?project=192763454488
View Tensorboard:
https://us-central1.tensorboard.googleusercontent.com/experiment/projects+192763454488+locations+us-central1+tensorboards+4361791214689714176+experiments+257110881370898432
Submitted Custom Job: projects/192763454488/locations/us-central1/customJobs/257110881370898432
MODEL_OUTPUT_DIR: gs://qwiklabs-asl-03-7c1aaee9a503/project_completion/trained_dnn_model_20260610_160652
Vertex AI TensorBoard: projects/192763454488/locations/us-central1/tensorboards/4361791214689714176
